# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library, including programmatic interaction with the dataset via its Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All elements are referenced by their `@id`.

In [ ]:
# List the record sets via their @id
print("Available record sets (by @id):")
for record_set in metadata.record_sets:
    print(f"- {record_set['@id']}: {record_set['name'] if 'name' in record_set else ''}")

# For this dataset, let's fetch the first record set and examine its fields/columns
record_sets = list(metadata.record_sets)
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    print(f"\nFields for record set @id='{main_record_set_id}':")
    for field in record_sets[0].get('fields', []):
        print(f"  - @id={field['@id']}  name={field.get('name', '')}  dataType={field.get('dataType', '')}")
    # For column display
    if 'columns' in record_sets[0]:
        print(f"\nColumns for record set @id='{main_record_set_id}':")
        for col in record_sets[0]['columns']:
            print(f"  - @id={col['@id']}  name={col.get('name', '')}  dataType={col.get('dataType', '')}")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. All identifiers use the entity `@id`.

In [ ]:
# Extract all record set @ids
record_set_ids = [r['@id'] for r in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set @id='{record_set_id}' with shape {df.shape}")
        else:
            print(f"No records found for record set @id='{record_set_id}'.")
    except Exception as e:
        print(f"Error loading record set @id='{record_set_id}': {e}")

# Preview the first record set's DataFrame (if available)
if dataframes:
    primary_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in DataFrame for record set @id='{primary_record_set_id}':")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing fields, and grouping data. All references use the corresponding `@id`.

In [ ]:
from IPython.display import display

# Example: Choose a numeric field (by @id) for filtering/normalization.
record_set_id = primary_record_set_id
df = dataframes[record_set_id]

print("Field options in main DataFrame:")
print(df.columns.tolist())
# Replace the below field id with an actual appropriate field @id (column) for numeric analysis
numeric_field_id = None
for col in df.columns:
    # Heuristically pick a numeric field
    if (df[col].dtype == 'float64' or df[col].dtype == 'int64') and 'coverage' in col.lower():
        numeric_field_id = col
        break

# If no coverage field is found, pick the first numeric one
if numeric_field_id is None:
    for col in df.columns:
        if df[col].dtype == 'float64' or df[col].dtype == 'int64':
            numeric_field_id = col
            break
print(f"\nUsing field '@id'='{numeric_field_id}' for numeric field EDA.")

if numeric_field_id:
    threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
    # Filtering
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (by @id)
    # Heuristically pick a categorical field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and (df[col].dtype == 'object' or str(df[col].dtype).startswith('category')):
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id, dropna=True).mean(numeric_only=True)
        print(f"\nGrouped data (mean) by '{group_field_id}':")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=30)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    # If a group field was chosen, boxplot by group
    if group_field_id:
        plt.figure(figsize=(12, 5))
        filtered_df.boxplot(column=numeric_field_id, by=group_field_id, grid=False, rot=45)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrates the use of the `mlcroissant` library to explore and process record sets, fields, and data available through the FAIR² dataset package via Croissant schema.

- We examined dataset metadata and identified record set and field `@id`s.
- We loaded and displayed data using these identifiers, applying standard data filtering and normalization routines.
- Visualizations highlighted distribution and groupwise structure for one numeric field.

For additional analysis, refer to the documentation and explore other record sets or fields as needed.